In [1]:
!pip install fastapi uvicorn nest-asyncio pyngrok joblib --quiet

In [2]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import joblib

from fastapi import FastAPI
from pydantic import BaseModel

import nest_asyncio
from pyngrok import ngrok

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df_uni = pd.read_csv("universities_profile_filled.csv")
scaler = joblib.load("scaler.pkl")

print("Files loaded successfully")

Files loaded successfully


In [4]:
class EligibilityNet(nn.Module):
    def __init__(self, input_dim=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

In [5]:
model = EligibilityNet().to(device)
model.load_state_dict(torch.load("eligibility_model.pt", map_location=device))
model.eval()

print("Model loaded successfully")

Model loaded successfully


In [6]:
def predict_for_all_universities(user, min_prob=0.10):
    results = []

    for _, uni in df_uni.iterrows():

        X = np.array([
            user["SAT"], user["ACT"], user["IELTS"], user["TOEFL"], user["GPA"],
            uni["SAT_Min"], uni["ACT_Min"], uni["IELTS_Min"],
            uni["TOEFL_Min"], uni["Average_GPA"]
        ]).reshape(1, -1)

        X = scaler.transform(X)
        X = torch.tensor(X, dtype=torch.float32).to(device)

        with torch.no_grad():
            prob = torch.sigmoid(model(X)).item()

        if prob >= min_prob:
            results.append({
                "university": uni["University"],
                "eligibility_probability": round(prob, 4)
            })

    results = sorted(results, key=lambda x: x["eligibility_probability"], reverse=True)

    return results

In [7]:
app = FastAPI(title="FYP Eligibility API")

class StudentInput(BaseModel):
    SAT: float
    ACT: float
    IELTS: float
    TOEFL: float
    GPA: float
    min_prob: float = 0.10


@app.get("/")
def home():
    return {"message": "API is running"}


@app.post("/predict")
def predict(student: StudentInput):

    user_data = student.dict()
    min_prob = user_data.pop("min_prob")

    results = predict_for_all_universities(user_data, min_prob)

    return {
        "total_universities": len(results),
        "top_results": results[:20]
    }

In [12]:
from pyngrok import ngrok

ngrok.set_auth_token("3BcYvQs0i7EJMUaA7z3EejOCj3W_6UwVqxwRmF2rnX63VYg5F")

In [ ]:
import nest_asyncio
nest_asyncio.apply()
from pyngrok import ngrok
ngrok.set_auth_token("3BcYvQs0i7EJMUaA7z3EejOCj3W_6UwVqxwRmF2rnX63VYg5F")

# kill any previous tunnels (important)
!killall ngrok

public_url = ngrok.connect(8000)
print("Public URL:", public_url)

import uvicorn

# ✅ THIS IS THE CORRECT WAY IN COLAB
config = uvicorn.Config(app=app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

await server.serve()

Public URL: NgrokTunnel: "https://cashable-unfavourable-savannah.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [3582]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     119.157.140.18:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     119.157.140.18:0 - "GET /openapi.json HTTP/1.1" 200 OK


/tmp/ipykernel_3582/1896681010.py:20: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  user_data = student.dict()


INFO:     119.157.140.18:0 - "POST /predict HTTP/1.1" 200 OK
